# Module 9 – Fine-tuning LoRA/QLoRA de Qwen2-VL-2B (étape 1 : image -> texte lu)

**À exécuter sur Google Colab avec un GPU** (Exécution > Modifier le type d'exécution > GPU).
GPU T4 gratuit : fonctionne en 4 bits (QLoRA). GPU L4/A100 : mets `USE_BF16 = True` plus bas.

**Avant de lancer, dépose dans Colab (panneau Fichiers à gauche) :**
1. le dossier `dataset_ordonnances/` (celui du premier zip) ;
2. le dossier `data/test_difficile/` et `data/medicaments.json` du projet `module9_api`.

Ce notebook n'a **pas été exécuté par son auteur** : lis les messages d'erreur éventuels et adapte les chemins.

In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes qwen-vl-utils pillow

In [ ]:
import json, random, unicodedata, re, time, statistics, difflib
from pathlib import Path
import torch
from PIL import Image
from transformers import (Qwen2VLForConditionalGeneration, AutoProcessor,
                          BitsAndBytesConfig, Trainer, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

BASE = "Qwen/Qwen2-VL-2B-Instruct"
DS = Path("dataset_ordonnances")            # dataset d'entraînement
HARD = Path("test_difficile")               # jeu de test difficile (copié depuis module9_api/data/)
REF = Path("medicaments.json")              # référentiel (copié depuis module9_api/data/)
OUT = Path("lora_ordonnances")              # adaptateur produit
USE_BF16 = False                            # True sur L4/A100
assert torch.cuda.is_available(), "Active un GPU dans Colab"
print(torch.cuda.get_device_name(0))

## 1. Modèle de base en 4 bits + configuration LoRA

In [ ]:
dtype = torch.bfloat16 if USE_BF16 else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
processor = AutoProcessor.from_pretrained(BASE, min_pixels=256*28*28, max_pixels=768*28*28)
model = Qwen2VLForConditionalGeneration.from_pretrained(BASE, quantization_config=bnb, device_map="auto", torch_dtype=dtype)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()      # attendu : moins de 1 % des paramètres

## 2. Mesure AVANT entraînement (modèle de base, prompt seul)

In [ ]:
P1 = "Lis tout le texte visible sur cette image (ordonnance ou boîte de médicament). Réponds uniquement avec le texte, une ligne par ligne lue."

def norm(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", s.lower()).split())

def lev(a, b):
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca != cb)))
        prev = cur
    return prev[-1]

BRANDS = [norm(m["marque"]) for m in json.load(open(REF, encoding="utf-8"))]
def brands_in(text):
    # même logique que l'API : on rapproche chaque 1 ou 2 mots d'une marque du référentiel
    w, found = norm(text).split(), set()
    for n in (1, 2):
        for i in range(len(w)-n+1):
            c = difflib.get_close_matches(" ".join(w[i:i+n]), BRANDS, n=1, cutoff=0.78)
            if c: found.add(c[0])
    return found

@torch.no_grad()
def read(img_path):
    img = Image.open(img_path).convert("RGB")
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": P1}]}]
    prompt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = processor(text=[prompt], images=[img], return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=400, do_sample=False)
    return processor.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]

def evaluate(root, jsonl, limit=60, tag=""):
    rows = [json.loads(l) for l in open(root/jsonl, encoding="utf-8") if l.strip()][:limit]
    cers, hit, tot, lat = [], 0, 0, []
    model.eval()
    for r in rows:
        gold = json.load(open(root/"labels"/f"{r['id']}.json", encoding="utf-8"))
        t0 = time.time(); pred = read(root/r["image"]); lat.append(time.time()-t0)
        g = norm(gold["texte_lu"]); cers.append(lev(norm(pred), g)/max(1, len(g)))
        found = brands_in(pred)
        for m in gold["medicaments"]:
            tot += 1; hit += norm(m["nom"]) in found
    print(f"[{tag}] images {len(rows)} | CER {statistics.mean(cers):.3f} | rappel médicaments {hit}/{tot} = {hit/tot:.1%} | latence médiane {statistics.median(lat):.1f} s")

evaluate(HARD, "etape1_vision_all.jsonl", tag="BASE sans adaptation - jeu difficile")

## 3. Données d'entraînement (image -> texte lu)

In [ ]:
class VisionJsonl(torch.utils.data.Dataset):
    def __init__(self, path, root):
        self.rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]; self.root = root
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        img = Image.open(self.root/r["image"]).convert("RGB")
        answer = r["messages"][1]["content"][0]["text"]
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": P1}]}]
        prompt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        full = processor(text=[prompt + answer + "<|im_end|>"], images=[img], return_tensors="pt")
        n_prompt = processor(text=[prompt], images=[img], return_tensors="pt")["input_ids"].shape[1]
        labels = full["input_ids"].clone(); labels[:, :n_prompt] = -100      # on n'apprend que la réponse
        item = {k: v[0] for k, v in full.items() if k not in ("pixel_values", "image_grid_thw")}
        item["pixel_values"] = full["pixel_values"]; item["image_grid_thw"] = full["image_grid_thw"][0]
        item["labels"] = labels[0]
        return item

def collate(batch):                      # batch_size = 1 : on évite le padding d'images
    b = batch[0]
    return {"input_ids": b["input_ids"][None], "attention_mask": b["attention_mask"][None], "labels": b["labels"][None],
            "pixel_values": b["pixel_values"], "image_grid_thw": b["image_grid_thw"][None]}

train_ds = VisionJsonl(DS/"etape1_vision_train.jsonl", DS)
val_ds = VisionJsonl(DS/"etape1_vision_val.jsonl", DS)
print(len(train_ds), "train |", len(val_ds), "val")

## 4. Entraînement LoRA (2 époques)

In [ ]:
args = TrainingArguments(
    output_dir="runs", num_train_epochs=2, per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05, weight_decay=0.0,
    fp16=not USE_BF16, bf16=USE_BF16, logging_steps=5, eval_strategy="epoch", save_strategy="no",
    gradient_checkpointing=True, remove_unused_columns=False, report_to="none", dataloader_num_workers=0)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)
model.config.use_cache = False
trainer.train()
model.config.use_cache = True
model.save_pretrained(OUT); processor.save_pretrained(OUT)
print("Adaptateur sauvegardé dans", OUT)

## 5. Mesure APRÈS entraînement

In [ ]:
evaluate(HARD, "etape1_vision_all.jsonl", tag="LORA - jeu difficile")
evaluate(DS, "etape1_vision_test.jsonl", limit=36, tag="LORA - jeu facile (test)")

## 6. Récupérer l'adaptateur
Télécharge le dossier `lora_ordonnances` (clic droit > Télécharger, ou zippe-le) et place-le dans `module9_api/models/lora_ordonnances`.
Puis : `OCR_ENGINE=lora uvicorn app.main:app` (voir README).

**Note pour le rapport** : recopie les trois lignes `[...]` imprimées ci-dessus dans le tableau de la section Évaluation.